# Spotify Popularity Prediction — CISC 4631
**Patch Shields | Group 3**

### Research Questions
1. Can audio features predict whether a song will be **globally popular**?
2. Can audio features predict whether a song will be popular **within its genre**?
3. Do the same features drive both, or does genre context change what matters?

### Pipeline
1. Load & Explore
2. Sample (stratified, balanced)
3. Preprocess & Engineer Labels
4. Baseline Classifiers (KNN, Decision Tree, Naive Bayes)
5. Neural Network (PyTorch MLP)
6. Compare & Analyze

# 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

import warnings 
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)

# 1. Load Data

In [ ]:
df = pd.read_csv('data/songs.csv')
print(f'Loaded {len(df):,}songs | {df.shape[1]} columns')
df.head()

# 2. Exploration

In [ ]:
# -- 2.1 Basic Info -- 
print(df.info())
print('\nMissing Values:')
missing = df.isnull().sum()
print(missing[missing > 0])

In [ ]:
# -- 2.2 Popularity Distribution --
fig, axes = plt.subplots(1,2,figsize=(12,4))

axes[0].hist(df['popularity'],bins=50, color='steelblue',edgecolor='white')
axes[0].set_title('Global Popularity Distribution')
axes[0].set_xlabel('Popularity (0-100)')
axes[0].set_ylabel('Count')

axes[1].hist(df['popularity'],bins=50,color='steelblue',edgecolor='white',cumulative=True,density=True)
axes[1].set_title('Cumulative Distribution')
axes[1].set_xlabel('Popularity (0-100)')

plt.tight_layout()
plt.show()

print(df['popularity'].describe())
print(f'\nSongs with 0 popularity: {(df['popularity']==0).sum():,} ({(df['popularity']==0).mean()*100:.1f}%)')

In [ ]:
# -- 2.3 Year Distribution ---
if 'year' in df.columns:
    df['year'].value_counts().sort_index().plot(kind='bar',figsize=(14,4),color='steelblue')
    plt.title('Songs per Year')
    plt.xlabel('Year')
    plt.tight_layout()
    plt.show()
    print(f'Songs from 2010 onward: {(df['year']>=2010).sum():,}')

In [ ]:
# -- 2.4 Genre Distribution --
if 'genre' in df.columns:
    top_genres = df['genre'].value_counts().head(20)
    top_genres.plot(kind='barh',figsize=(10,6),color='steelblue')
    plt.title('Top 20 Genres by Song Count')
    plt.tight_layout()
    plt.show()
    print(f'Total Unique Genres: {df['genre'].nunique()}')
    


In [ ]:
# -- 2.5 Popularity within Top Genres --
# Question: does popularity distribution vary by genre?
if 'genre' in df.columns:
    top_10_genres = df['genre'].value_counts().head(10).index
    fig, axes = plt.subplot(2,5,figsize=(18,7),sharey=False)

    for ax, genre in zip(axes.flatten(), top_10_genres):
        subset = df[df['genre'] == genre]['popularity']
        ax.hist(subset,bins=30,color='steelblue',edgecolor='white')
        ax.set_title(genre[:20],fontsize=9)
        ax.set_xlabel('Popularity')
    
    plt.suptitle('Popularity Distribution by Genre',y=1.02)
    plt.tight_layout()
    plt.show()

In [ ]:
# -- 2.6 Feature Correlation with Popularity --
AUDIO_FEATURES = [
    'danceability','energy','loudness',
    'speechiness','acousticness','instrumentalness',
    'liveness','valence','tempo','duration_ms'
]
valid_features = [f for f in AUDIO_FEATURES if f in df.columns]

corr = df[valid_features + ['popularity']].corr()['popularity'].drop('popularity').sort_vaules()
corr.plot(kind='barh',figsize=(8,5),color='steelblue')
plt.title('Audio Feature Correlation with Popularity')
plt.axvline(0,color='black',linewidth=0.8)
plt.tight_layout()
plt.show()